# 成员 2：ResNet50 可解释性分析

研究问题：ResNet50 与 MobileNetV3-Large 在相同图片上的解释区域是否相似？本 Notebook 记录 ResNet50 的 Grad-CAM、Integrated Gradients（IG）和 LIME 实验。

数据来自 `member1_imagenet_handoff/`。主分析解释 ResNet50 自己的 top-1 预测；B/C/D 另生成共同真实类别目标，供跨模型配对。


## 实验设置

- ResNet50：`ResNet50_Weights.IMAGENET1K_V1`；使用权重自带 transforms（短边 resize 256、center crop 224、ImageNet mean/std）。
- 输入从 `images/` 原图生成；`input_crops/` 只用于查看裁剪范围。
- 目标：`pred` 为 ResNet50 自己的 top-1；`common_gt` 为真实标签。B 组 ResNet50 预测正确，因此两种目标相同。
- 全组另一模型为 MobileNetV3-Large V1；成员 3 使用同一批图片和对应目标策略。


## 环境

已在项目的 `BYtorch` 环境运行。当前依赖版本见代码输出；其他环境仅在缺少 Captum、LIME 或 scikit-image 时安装相应依赖。


In [ ]:
# 如当前环境缺少依赖，取消注释后运行：
# %pip install captum lime scikit-image


In [ ]:
from pathlib import Path
from importlib.metadata import version
import csv
import hashlib
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision
from captum.attr import IntegratedGradients
from lime import lime_image
from skimage.segmentation import mark_boundaries
from torchvision.models import ResNet50_Weights, resnet50

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

PROJECT_ROOT = Path.cwd()
HANDOFF_ROOT = PROJECT_ROOT / "member1"
MANIFEST_PATH = HANDOFF_ROOT / "manifest.json"
PREDICTIONS_PATH = HANDOFF_ROOT / "predictions.csv"
OUTPUT_DIR = PROJECT_ROOT / "member2/outputs/resnet50"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Captum:", version("captum"))
print("LIME:", version("lime"))
print("scikit-image:", version("scikit-image"))
print("Device:", DEVICE)
print("Handoff:", HANDOFF_ROOT.resolve())


In [ ]:
WEIGHTS = ResNet50_Weights.IMAGENET1K_V1
MODEL = resnet50(weights=WEIGHTS).to(DEVICE).eval()
PREPROCESS = WEIGHTS.transforms()
CATEGORIES = WEIGHTS.meta["categories"]
MEAN = torch.tensor(PREPROCESS.mean, dtype=torch.float32).view(1, 3, 1, 1)
STD = torch.tensor(PREPROCESS.std, dtype=torch.float32).view(1, 3, 1, 1)

print("Weights:", WEIGHTS)
print("Preprocess:", PREPROCESS)
print("Number of output classes:", len(CATEGORIES))


## 数据清单

数据由成员 1 的交接包提供。本 Notebook 会按 `image_id` 合并 `manifest.json` 与 `predictions.csv`，检查两份清单的 ID 一致、图片存在，并核对图片 SHA256。若从其他目录启动 Notebook，请将 `PROJECT_ROOT` 改为本项目目录。

| 字段 | 用途 |
|---|---|
| `image_id`、`group` | 统一样本编号及 A/B/C/D 分组 |
| `image_path` | 交接包原图的相对路径 |
| `ground_truth_id` | ImageNet/Torchvision 0–999 类别编号 |
| `resnet50_target_id` | 主分析目标：ResNet50 自身 top-1 预测 |
| `common_target_id` | B/C/D 补充分析目标：同一真实类别 |

正式推理使用 `images/` 原图和权重自带 transforms；`input_crops/` 仅供目视检查。


In [ ]:
def read_manifest():
    if not MANIFEST_PATH.exists() or not PREDICTIONS_PATH.exists():
        raise FileNotFoundError(
            "成员 1 交接文件缺失：需要 manifest.json 和 predictions.csv，"
            f"当前目录为 {HANDOFF_ROOT}"
        )

    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    manifest_by_id = {str(row["image_id"]): row for row in manifest}
    with PREDICTIONS_PATH.open(newline="", encoding="utf-8-sig") as f:
        prediction_rows = list(csv.DictReader(f))
    prediction_by_id = {row["image_id"].strip(): row for row in prediction_rows}

    if len(manifest_by_id) != len(manifest) or len(prediction_by_id) != len(prediction_rows):
        raise ValueError("manifest.json 或 predictions.csv 中存在重复 image_id")
    if manifest_by_id.keys() != prediction_by_id.keys():
        raise ValueError("manifest.json 与 predictions.csv 的 image_id 不一致")

    rows = []
    for sample_id, metadata in manifest_by_id.items():
        prediction = prediction_by_id[sample_id]
        image_path = HANDOFF_ROOT / metadata["image_path"]
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        if digest != metadata["sha256"]:
            raise ValueError(f"图片校验失败：{sample_id} ({image_path})")
        if prediction["image_path"] != metadata["image_path"]:
            raise ValueError(f"{sample_id} 的图片路径在两份清单中不一致")

        rows.append({
            "sample_id": sample_id,
            "group": prediction["group"],
            "image_path": str(image_path),
            "true_class_idx": int(prediction["ground_truth_id"]),
            "true_class": prediction["ground_truth_class"],
            "resnet50_pred_id": int(prediction["resnet50_pred_id"]),
            "resnet50_pred_class": prediction["resnet50_pred_class"],
            "resnet50_confidence": float(prediction["resnet50_confidence"]),
            "resnet50_target_id": int(prediction["resnet50_target_id"]),
            "common_target_id": int(prediction["common_target_id"]),
            "common_target_class": prediction["common_target_class"],
            "supplement_common_target": prediction["supplement_common_target"].lower() == "true",
        })
    return sorted(rows, key=lambda row: row["sample_id"])


SAMPLES = read_manifest()
print("Number of samples:", len(SAMPLES))
print("Groups:", {group: sum(row["group"] == group for row in SAMPLES) for group in "ABCD"})


In [ ]:
def load_rgb(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return Image.open(path).convert("RGB")


def model_input_and_display(image):
    # Return the normalized model input and its matching RGB crop for plotting.
    x = PREPROCESS(image).unsqueeze(0)
    display = (x * STD + MEAN).squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
    return x.to(DEVICE), display


@torch.inference_mode()
def predict_tensor(x):
    logits = MODEL(x.to(DEVICE))
    probs = logits.softmax(dim=1)
    score, index = probs.max(dim=1)
    index = int(index.item())
    return {
        "pred_class_idx": index,
        "pred_class": CATEGORIES[index],
        "confidence": float(score.item()),
        "probabilities": probs.squeeze(0).cpu().numpy(),
    }


## Grad-CAM

对指定类别的梯度在卷积特征图上加权，得到空间热图。本实验目标层为 ResNet50 的 `layer4[-1]`。


In [ ]:
def gradcam(x, target_idx):
    target_layer = MODEL.layer4[-1]
    captured = {}

    def save_activation(_module, _inputs, output):
        captured["activation"] = output
        output.register_hook(lambda grad: captured.__setitem__("gradient", grad))

    handle = target_layer.register_forward_hook(save_activation)
    MODEL.zero_grad(set_to_none=True)
    try:
        logits = MODEL(x.to(DEVICE))
        logits[0, target_idx].backward()
        activation = captured["activation"].detach()
        gradient = captured["gradient"].detach()
        channel_weights = gradient.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((channel_weights * activation).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0, 0]
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy()
    finally:
        handle.remove()
        MODEL.zero_grad(set_to_none=True)


## Integrated Gradients（IG）

沿 baseline 到输入的路径累计像素梯度。使用归一化空间全零 baseline（对应 ImageNet 平均颜色）和 64 个积分步；图中正负颜色表示对目标分数的正/负归因。记录 convergence delta 作为数值近似诊断。


In [ ]:
IG_STEPS = 64


def integrated_gradients(x, target_idx, n_steps=IG_STEPS):
    explainer = IntegratedGradients(MODEL)
    baseline = torch.zeros_like(x)
    attribution, delta = explainer.attribute(
        x.to(DEVICE),
        baselines=baseline.to(DEVICE),
        target=target_idx,
        n_steps=n_steps,
        return_convergence_delta=True,
    )
    # Sum signed channel contributions to retain positive/negative evidence.
    heat = attribution.sum(dim=1)[0].detach().cpu().numpy()
    return heat, float(delta.detach().abs().mean().cpu())


## LIME

对 224×224 输入裁剪进行超像素扰动，以局部近似解释目标类别。设置：quickshift 默认分割、500 次扰动、最多显示 10 个超像素、黑色遮挡、batch size 32、seed 2026；图中保留正负区域。


In [ ]:
LIME_NUM_SAMPLES = 500
LIME_NUM_FEATURES = 10


def lime_classifier(images):
    batch = torch.as_tensor(np.asarray(images), dtype=torch.float32)
    if batch.max() > 1.0:
        batch = batch / 255.0
    batch = batch.permute(0, 3, 1, 2)
    mean = MEAN.to(batch.device)
    std = STD.to(batch.device)
    batch = (batch - mean) / std
    with torch.inference_mode():
        return MODEL(batch.to(DEVICE)).softmax(dim=1).cpu().numpy()


def lime_explanation(display_crop, target_idx, num_samples=LIME_NUM_SAMPLES):
    image = np.uint8(np.clip(display_crop, 0, 1) * 255)
    explainer = lime_image.LimeImageExplainer(random_state=SEED)
    explanation = explainer.explain_instance(
        image,
        lime_classifier,
        labels=(target_idx,),
        hide_color=0,
        num_samples=num_samples,
        batch_size=32,
    )
    lime_img, mask = explanation.get_image_and_mask(
        target_idx,
        positive_only=False,
        num_features=LIME_NUM_FEATURES,
        hide_rest=False,
    )
    return lime_img, mask, explanation


## 运行

默认不自动重复实验。手动设定 `SAMPLE_ID` 后可单跑；设批量单元中的 `RUN_FULL_EXPERIMENT=True` 可重跑 20 个 pred 和 B/C/D 的 15 个 common_gt 条目。


In [ ]:
SAMPLE_ID = ""
TARGET_MODE = "pred"  # "pred" 主分析；"common_gt" 用于 B/C/D 补充分析


def run_sample(sample_id, target_mode=TARGET_MODE, display=True):
    row = next((r for r in SAMPLES if r["sample_id"] == sample_id), None)
    if row is None:
        raise KeyError(f"Sample ID not found: {sample_id}")
    if target_mode not in {"pred", "common_gt"}:
        raise ValueError("target_mode 必须是 'pred' 或 'common_gt'")
    if target_mode == "common_gt" and row["group"] not in {"B", "C", "D"}:
        raise ValueError("共同真实类别补充分析仅适用于 B/C/D 组")

    image = load_rgb(row["image_path"])
    x, display_crop = model_input_and_display(image)
    prediction = predict_tensor(x)
    if target_mode == "pred":
        target_idx = row["resnet50_target_id"]
        target_source = "ResNet50 own top-1 prediction"
    else:
        target_idx = row["common_target_id"]
        target_source = "shared ground-truth target"

    cam = gradcam(x, target_idx)
    ig_map, ig_delta = integrated_gradients(x, target_idx)
    lime_img, lime_mask, lime_exp = lime_explanation(display_crop, target_idx)

    fig, axes = plt.subplots(1, 4, figsize=(17, 4))
    axes[0].imshow(display_crop)
    axes[0].set_title(f"Input crop\n{prediction['pred_class']} ({prediction['confidence']:.3f})")
    axes[1].imshow(display_crop)
    axes[1].imshow(cam, cmap="jet", alpha=0.45)
    axes[1].set_title("Grad-CAM")
    scale = np.max(np.abs(ig_map)) + 1e-8
    axes[2].imshow(display_crop)
    axes[2].imshow(ig_map / scale, cmap="seismic", vmin=-1, vmax=1, alpha=0.55)
    axes[2].set_title(f"IG (steps={IG_STEPS})\n|delta|={ig_delta:.4g}")
    axes[3].imshow(mark_boundaries(lime_img / 255.0, lime_mask))
    axes[3].set_title("LIME")
    for ax in axes:
        ax.axis("off")
    target_label = "pred" if target_mode == "pred" else "gt"
    fig.suptitle(
        f"{sample_id} ({row['group']}) | true={row['true_class']} | "
        f"target={CATEGORIES[target_idx]} ({target_idx}) | {target_source}"
    )
    fig.tight_layout()
    output_path = OUTPUT_DIR / f"{sample_id}_resnet50_xai_{target_label}.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    if display:
        plt.show()
    else:
        plt.close(fig)

    return {
        "sample_id": sample_id,
        "group": row["group"],
        "true_class_idx": row["true_class_idx"],
        "true_class": row["true_class"],
        "target_mode": target_mode,
        "target_class_idx": target_idx,
        "target_class": CATEGORIES[target_idx],
        "target_source": target_source,
        "pred_class_idx": prediction["pred_class_idx"],
        "pred_class": prediction["pred_class"],
        "confidence": prediction["confidence"],
        "correct": row["true_class_idx"] == prediction["pred_class_idx"],
        "ig_convergence_delta_abs_mean": ig_delta,
        "figure": str(output_path),
    }


if SAMPLE_ID:
    RESULT = run_sample(SAMPLE_ID)
    RESULT


In [ ]:
RUN_FULL_EXPERIMENT = False


def run_dataset():
    results = []
    for row in SAMPLES:
        results.append(run_sample(row["sample_id"], target_mode="pred", display=False))
        if row["group"] in {"B", "C", "D"}:
            results.append(run_sample(row["sample_id"], target_mode="common_gt", display=False))

    result_path = OUTPUT_DIR / "resnet50_xai_results.csv"
    columns = [
        "sample_id", "group", "true_class_idx", "true_class", "target_mode",
        "target_class_idx", "target_class", "pred_class_idx", "pred_class",
        "confidence", "correct", "ig_convergence_delta_abs_mean", "figure",
    ]
    with result_path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows({key: result[key] for key in columns} for result in results)
    print(f"Saved {len(results)} explanation runs to {result_path}")
    return results


if RUN_FULL_EXPERIMENT:
    BATCH_RESULTS = run_dataset()


## 结果与初步发现

实验完成 20 个主分析目标（`pred`）及 B/C/D 的 15 个共同真实类别目标（`common_gt`），共 35 张拼图。成员 1 的分组中，ResNet50 在 A、B 共 10 张正确，在 C、D 共 10 张错误；该样本集按组平衡挑选，不能据此估计总体准确率。完整数值见 [结果 CSV](results/resnet50_xai_results.csv)，35 张图在本地实验目录生成；公开包省略这些含 ImageNet 内容的图像文件，重跑后可在 `member2/outputs/resnet50/` 查看。

### 代表案例

**A01：正确预测 sea snake，confidence=0.561。**Grad-CAM 的高响应覆盖蛇身及邻近区域，LIME 也选中蛇身附近的超像素；IG 显示更细碎的正负贡献。三种方法都涉及主体附近，但空间粒度和背景范围不同。|IG delta|=0.217。


**B01：ResNet50 正确预测 Shetland sheepdog，confidence=0.739；MobileNetV3 预测 collie。**ResNet50 的 Grad-CAM 和 LIME 都突出狗脸/头部附近；IG 分布较散，且 |delta|=0.822，故此例的 IG 数值近似尤其需要谨慎。它可以与成员 3 的同图结果配对，但不能仅凭这张图推断模型为何区分两个相近犬类。


**C01：真实类别 table lamp，ResNet50 以 confidence=0.998 预测 guillotine。**在 `pred` 图中，Grad-CAM 响应集中于中央支架；改用 `common_gt` 解释 table lamp 后，热图分布改变，较强响应更多出现在底座/桌面附近。模型输出仍是 guillotine，gt 只是改变了归因目标。原始图的灯罩大部分不在 224×224 中心裁剪中，因此裁剪可能是调查线索；本实验没有通过改变裁剪来验证因果关系。



**D03：真实类别 garter snake，ResNet50 预测 sea snake，confidence=0.395。**Grad-CAM 高响应覆盖蛇身附近，但模型仍错在具体蛇类；LIME 选区较大，也包含背景。关注到物体附近不等于模型正确识别其细分类。


### 跨样本观察与限制

- Grad-CAM 给出较平滑、低分辨率的空间线索；IG 给出像素级正负归因；LIME 依赖超像素扰动。三者互补，但图上的颜色或区域不能跨方法直接比较强弱。
- 20 张 pred 结果的 IG |delta| 中位数约 0.292，7 张大于 0.5；这不是准确率，而是数值近似诊断。增步诊断仅核查 B02、C05、D01，未对所有样本复算正式归因图。
- B 组 ResNet50 预测正确，故其 `common_gt` 与 `pred` 目标相同。保留 B-gt 输出是为了和 MobileNetV3 的真实标签解释配对，不是 ResNet50 的额外独立证据。
- A/B/C/D 各选 5 张，样本有意平衡。错误案例中的可视化只能支持对这些图的观察，不能证明模型依赖某特征或解释错误原因。
- 当前仅有 ResNet50 结果；是否两个模型关注相似区域，要等成员 3 的配对结果和成员 4 的新方法后再回答。成员 5 可从上面案例中选图整合，但需把最终模型间结论留待全组证据确认。

交接细节见[给成员 5 的交接文档](Assignment2_成员2_交接说明.md)。
